# Ingest Human Frame Annotations

This notebook reads completed protected XLSX annotation workbooks, checks hierarchical label validity and workbook integrity, derives frame labels, and writes clean machine-readable label tables for the classifier stage.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from openpyxl import load_workbook

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CLASSIFICATION_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
HUMAN_DIR = CLASSIFICATION_DIR / "human_annotation"
OUTPUT_DIR = CLASSIFICATION_DIR / "human_labels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_COMPLETED = HUMAN_DIR / "frame_pilot_annotation_completed.xlsx"
VALIDATION_COMPLETED = HUMAN_DIR / "frame_validation_annotation_completed.xlsx"

EXPECTED_COLUMNS = [
    "annotation_id",
    "context_id",
    "analysis_unit",
    "lsc_year",
    "raw_form",
    "target_sentence_plus_adjacent",
    "substantive_target_discourse",
    "clinical_frame_present",
    "lived_experience_frame_present",
    "confidence",
    "annotation_round",
    "codebook_version",
]
EXPECTED_ROUND_ROWS = {"pilot": 200, "validation": 200}
CONFIDENCE_VALUES = {"high", "medium", "low"}
SPREADSHEET_ERROR_VALUES = {"#NAME?", "#VALUE!", "#REF!", "#DIV/0!", "#NUM!", "#NULL!", "#N/A", "#SPILL!", "#CALC!"}


## Load Completed Workbooks

Complete the protected `annotations` sheet and save each workbook with the `_completed.xlsx` filename before running this notebook. Do not export or resave the annotation handoff as CSV through Excel.


In [ ]:
missing = [path for path in [PILOT_COMPLETED, VALIDATION_COMPLETED] if not path.exists()]
if missing:
    print("Completed annotation workbooks not found yet:")
    for path in missing:
        print(f"- {path.relative_to(PROJECT_ROOT)}")
    raise SystemExit("Add completed XLSX workbooks, then rerun.")


def inspect_workbook_cells(path: Path) -> list[dict]:
    workbook = load_workbook(path, read_only=True, data_only=False)
    if "annotations" not in workbook.sheetnames:
        raise ValueError(f"Missing annotations sheet: {path.name}")
    worksheet = workbook["annotations"]
    issues = []
    for row in worksheet.iter_rows():
        for cell in row:
            if cell.data_type == "f":
                issues.append({"workbook": path.name, "cell": cell.coordinate, "issue": "formula", "value": cell.value})
            elif cell.data_type == "e" or str(cell.value).strip() in SPREADSHEET_ERROR_VALUES:
                issues.append({"workbook": path.name, "cell": cell.coordinate, "issue": "spreadsheet_error", "value": cell.value})
    workbook.close()
    return issues


workbook_issues = []
for path in [PILOT_COMPLETED, VALIDATION_COMPLETED]:
    workbook_issues.extend(inspect_workbook_cells(path))
if workbook_issues:
    raise ValueError(f"Formula or spreadsheet-error cells detected: {workbook_issues[:20]}")

pilot = pd.read_excel(PILOT_COMPLETED, sheet_name="annotations", dtype=str, keep_default_na=False)
validation = pd.read_excel(VALIDATION_COMPLETED, sheet_name="annotations", dtype=str, keep_default_na=False)
human = pd.concat([pilot, validation], ignore_index=True)

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(human.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")
human = human[EXPECTED_COLUMNS].copy()

for round_name, frame in [("pilot", pilot), ("validation", validation)]:
    expected_rows = EXPECTED_ROUND_ROWS[round_name]
    if len(frame) != expected_rows:
        raise ValueError(f"Expected {expected_rows} {round_name} rows, found {len(frame)}.")
    if set(frame["annotation_round"].astype(str).str.strip()) != {round_name}:
        raise ValueError(f"Unexpected annotation_round values in {round_name} workbook.")

spreadsheet_error_cells = []
for column in EXPECTED_COLUMNS:
    error_rows = human.loc[human[column].astype(str).str.strip().isin(SPREADSHEET_ERROR_VALUES), ["annotation_id", column]]
    if not error_rows.empty:
        spreadsheet_error_cells.extend(error_rows.to_dict("records"))
if spreadsheet_error_cells:
    raise ValueError(f"Spreadsheet error values detected after workbook load: {spreadsheet_error_cells[:20]}")

print(f"Loaded human annotations: {len(human):,}")


## Validate and Derive Frames

The Stage-0 sufficiency gate is labelled for every row. Clinical and lived-experience labels are required only for substantive rows and must be `NA` for non-substantive rows.


In [ ]:
def parse_bool_or_na(value: object) -> object:
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    if text in {"", "na", "n/a", "none", "nan"}:
        return pd.NA
    if text in {"true", "t", "yes", "y", "1"}:
        return True
    if text in {"false", "f", "no", "n", "0"}:
        return False
    return pd.NA

for column in ["substantive_target_discourse", "clinical_frame_present", "lived_experience_frame_present"]:
    human[column] = human[column].map(parse_bool_or_na).astype("boolean")

invalid_substantive = human.loc[human["substantive_target_discourse"].isna(), ["annotation_id", "substantive_target_discourse"]]
if not invalid_substantive.empty:
    raise ValueError(f"Invalid or missing Stage-0 labels: {invalid_substantive.head(20).to_dict('records')}")

substantive = human["substantive_target_discourse"].eq(True)
non_substantive = human["substantive_target_discourse"].eq(False)
invalid_stage1_substantive = human.loc[
    substantive & (human["clinical_frame_present"].isna() | human["lived_experience_frame_present"].isna()),
    ["annotation_id", "clinical_frame_present", "lived_experience_frame_present"],
]
if not invalid_stage1_substantive.empty:
    raise ValueError(f"Missing Stage-1 labels for substantive rows: {invalid_stage1_substantive.head(20).to_dict('records')}")

invalid_stage1_non_substantive = human.loc[
    non_substantive & (human["clinical_frame_present"].notna() | human["lived_experience_frame_present"].notna()),
    ["annotation_id", "clinical_frame_present", "lived_experience_frame_present"],
]
if not invalid_stage1_non_substantive.empty:
    raise ValueError(f"Clinical/lived labels must be NA for non-substantive rows: {invalid_stage1_non_substantive.head(20).to_dict('records')}")

human["confidence"] = human["confidence"].astype(str).str.strip().str.lower()
invalid_confidence = human.loc[~human["confidence"].isin(CONFIDENCE_VALUES), ["annotation_id", "confidence"]]
if not invalid_confidence.empty:
    raise ValueError(f"Invalid confidence values: {invalid_confidence.head(20).to_dict('records')}")


def derive_frame(row: pd.Series) -> str:
    if not bool(row["substantive_target_discourse"]):
        return "non_substantive_or_insufficient"
    clinical = bool(row["clinical_frame_present"])
    lived = bool(row["lived_experience_frame_present"])
    if clinical and lived:
        return "mixed"
    if clinical:
        return "clinical_only"
    if lived:
        return "lived_only"
    return "substantive_other"

human["derived_frame"] = human.apply(derive_frame, axis=1)

duplicate_ids = human["annotation_id"].duplicated().sum()
duplicate_contexts = human["context_id"].duplicated().sum()
if duplicate_ids or duplicate_contexts:
    raise ValueError(f"Duplicate IDs found: annotation_id={duplicate_ids}, context_id={duplicate_contexts}")

human.groupby(["annotation_round", "analysis_unit", "derived_frame"]).size()


## Save Clean Human Labels

In [ ]:
human_path = OUTPUT_DIR / "frame_human_labels.csv"
pilot_path = OUTPUT_DIR / "frame_human_pilot_labels.csv"
validation_path = OUTPUT_DIR / "frame_human_validation_labels.csv"

human.to_csv(human_path, index=False)
human.loc[human["annotation_round"].eq("pilot")].to_csv(pilot_path, index=False)
human.loc[human["annotation_round"].eq("validation")].to_csv(validation_path, index=False)

print("Wrote clean human labels:")
for path in [human_path, pilot_path, validation_path]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")